# RAG Agent
This notebook provides an example of a simple Retrieval-Augmented Generation (RAG) agent using the inference adaptor, Vector I/O adaptors, inline embeddings, and the agent adaptor.

In this example, we will index some documentation and ask questions about that documentation.

The tool we use is the memory tool. Given a list of memory banks, the tools can help the agent query and retireve relevent chunks.
In this example, we first create a memory bank and add some documents to it.
Then we configure the agent to use the memory tool.

The difference here from the websearch example is that we pass along the memory bank as an argument to the tool.

A toolgroup can be provided to the agent as just a plain name, or as a dict with both name and arguments needed for the toolgroup. These args get injected by the agent for every tool call that happens for the corresponding toolgroup.

Please refer to the [OGX documentation on RAG](https://ogx-ai.github.io/docs/building_applications/rag) for further details.

## Setup

In [ ]:
# Imports
import os
from io import BytesIO

import requests
from ogx_client import OgxClient

# Select model
model = "sambanova/Meta-Llama-3.3-70B-Instruct"


In [2]:
# Create HTTP client
OGX_PORT = 8321
client = OgxClient(base_url=f"http://localhost:{OGX_PORT}")


In [ ]:
# Upload documents to a vector store
urls = ["chat.rst", "llama3.rst", "memory_optimizations.rst", "lora_finetune.rst"]
base_url = "https://raw.githubusercontent.com/pytorch/torchtune/main/docs/source/tutorials/"

file_ids = []
for url in urls:
    resp = requests.get(base_url + url)
    buf = BytesIO(resp.content)
    buf.name = url
    f = client.files.create(file=buf, purpose="assistants")
    file_ids.append(f.id)
    print(f"Uploaded {url} -> {f.id}")

vector_store = client.vector_stores.create(
    name="torchtune-docs",
    file_ids=file_ids,
    timeout=300,
)
print(f"Vector store: {vector_store.id}")


In [6]:
previous_response_id = None

user_prompts = [
    "What are the top 5 topics that were explained? Only list succinct bullet points.",
    "How do I optimize memory usage in torchtune?",
]

for prompt in user_prompts:
    print(f"\nUser> {prompt}")
    response = client.responses.create(
        model=model,
        instructions="You are a helpful assistant",
        input=prompt,
        tools=[{
            "type": "file_search",
            "vector_store_ids": [vector_store.id],
        }],
        max_tool_calls=5,
        previous_response_id=previous_response_id,
    )
    previous_response_id = response.id
    print(response.output_text)
